# Patch vs clean — highway brake-reaction analysis

Compare follower behavior over 10 clean vs 10 patched runs on Town04 highway.
- **Leader** (`vehicle.carlamotors.carlacola`) cruises at 40 km/h for 10 s, then hard-brakes for 5 s.
- **Follower** (`vehicle.tesla.model3`) driven by PCLA agent `tfv6_visiononly` — has to detect the leader and brake.
- The only difference between conditions: the rear panel of the leader. CLEAN = original CarlaCola red; PATCH = the adversarial TGA we trained against YOLO.

Metrics extracted: brake **reaction time** (when does follower brake > 0.2?), **minimum distance**, **collision count**, **TTC** profile.

In [ ]:
import json, re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path('/home/vortex/adversarial-patch-vehicle') if Path('/home/vortex/adversarial-patch-vehicle').exists() else Path.cwd()
RUN_ROOT = REPO_ROOT / 'experiments/carla_scenarios/patch_vs_clean_20260609_115057'
CLEAN_DIR = RUN_ROOT / 'clean'
PATCH_DIR = RUN_ROOT / 'patch'
BRAKE_START_S = 10.0   # leader brakes at 10 s
BRAKE_THRESH = 0.2     # follower considered 'reacting' if brake > this
print('Clean runs:', len(list(CLEAN_DIR.glob('*/telemetry.csv'))))
print('Patch runs:', len(list(PATCH_DIR.glob('*/telemetry.csv'))))

In [ ]:
def load_runs(label, root):
    out = []
    for d in sorted(root.glob('*/telemetry.csv')):
        run = d.parent
        tel = pd.read_csv(run / 'telemetry.csv')
        agt = pd.read_csv(run / 'agent.csv')
        summary = json.loads((run / 'summary.json').read_text())
        out.append({'label': label, 'run_dir': run, 'telemetry': tel, 'agent': agt, 'summary': summary})
    return out

clean_runs = load_runs('clean', CLEAN_DIR)
patch_runs = load_runs('patch', PATCH_DIR)
print(f'Loaded {len(clean_runs)} clean and {len(patch_runs)} patch runs')

In [ ]:
def per_run_metrics(runs):
    rows = []
    for r in runs:
        tel, agt, s = r['telemetry'], r['agent'], r['summary']
        # Brake reaction time: first sim_time after BRAKE_START_S where brake > BRAKE_THRESH
        post = agt[agt['sim_time_s'] >= BRAKE_START_S]
        react = post[post['brake'] > BRAKE_THRESH]
        reaction_time = (react['sim_time_s'].iloc[0] - BRAKE_START_S) if len(react) > 0 else None
        # Min distance after leader brake start
        post_tel = tel[tel['sim_time_s'] >= BRAKE_START_S]
        min_dist = float(post_tel['distance_m'].min()) if len(post_tel) else None
        # Collision
        collisions = int(tel['collision_detected'].sum())
        # End follower speed
        end_follower_kmh = float(tel['follower_speed_kmh'].iloc[-1])
        rows.append({
            'run': r['run_dir'].name,
            'reaction_time_s': reaction_time,
            'min_distance_m': min_dist,
            'collisions': collisions,
            'follower_speed_end_kmh': end_follower_kmh,
        })
    return pd.DataFrame(rows)

clean_metrics = per_run_metrics(clean_runs)
patch_metrics = per_run_metrics(patch_runs)
clean_metrics['label'] = 'clean'
patch_metrics['label'] = 'patch'
all_metrics = pd.concat([clean_metrics, patch_metrics], ignore_index=True)
all_metrics

In [ ]:
summary = all_metrics.groupby('label').agg(
    n_runs=('run', 'count'),
    reaction_time_mean=('reaction_time_s', 'mean'),
    reaction_time_median=('reaction_time_s', 'median'),
    min_dist_mean_m=('min_distance_m', 'mean'),
    min_dist_min_m=('min_distance_m', 'min'),
    total_collisions=('collisions', 'sum'),
    runs_with_collision=('collisions', lambda s: int((s > 0).sum())),
    follower_end_speed_mean=('follower_speed_end_kmh', 'mean'),
).round(3)
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Distance over time per run
for r in clean_runs:
    axes[0,0].plot(r['telemetry']['sim_time_s'], r['telemetry']['distance_m'], color='green', alpha=0.4)
for r in patch_runs:
    axes[0,0].plot(r['telemetry']['sim_time_s'], r['telemetry']['distance_m'], color='red', alpha=0.4)
axes[0,0].axvline(BRAKE_START_S, color='k', ls='--', alpha=0.5)
axes[0,0].set_xlabel('sim time s'); axes[0,0].set_ylabel('distance m')
axes[0,0].set_title('Follower-leader distance (green=clean, red=patch)')

# Brake input per run
for r in clean_runs:
    axes[0,1].plot(r['agent']['sim_time_s'], r['agent']['brake'], color='green', alpha=0.4)
for r in patch_runs:
    axes[0,1].plot(r['agent']['sim_time_s'], r['agent']['brake'], color='red', alpha=0.4)
axes[0,1].axvline(BRAKE_START_S, color='k', ls='--', alpha=0.5)
axes[0,1].set_xlabel('sim time s'); axes[0,1].set_ylabel('follower brake input')
axes[0,1].set_title('Brake input over time')

# Reaction time histogram
axes[1,0].hist(clean_metrics['reaction_time_s'].dropna(), bins=15, alpha=0.6, color='green', label='clean')
axes[1,0].hist(patch_metrics['reaction_time_s'].dropna(), bins=15, alpha=0.6, color='red', label='patch')
axes[1,0].set_xlabel('reaction time s (after leader brakes)'); axes[1,0].set_ylabel('runs')
axes[1,0].set_title('Brake reaction time')
axes[1,0].legend()

# Min distance vs reaction time scatter
axes[1,1].scatter(clean_metrics['reaction_time_s'], clean_metrics['min_distance_m'], color='green', label='clean', s=60)
axes[1,1].scatter(patch_metrics['reaction_time_s'], patch_metrics['min_distance_m'], color='red', label='patch', s=60)
axes[1,1].axhline(0, color='k', ls='--', alpha=0.3)
axes[1,1].set_xlabel('reaction time s'); axes[1,1].set_ylabel('min distance m')
axes[1,1].set_title('Reaction vs outcome (lower min-dist = closer call)')
axes[1,1].legend()

plt.tight_layout(); plt.show()

## Interpretation

- If the patch is working downstream, in the **red runs** we expect: later reaction time, smaller min distance, more collisions.
- A **null result** (red ≈ green) means the patch reduces YOLO confidence but not enough to alter the PCLA agent's brake decision — the agent has its own internal redundancy (depth, motion cues).
- These results were obtained with PCLA agent `tfv6_visiononly`. Want stronger transfer? Re-run with `tfv4_aim` (vision-only Transfuser v4) and `simlingo_simlingo` (VLM).